In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.01', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54,-8.54
1,IIT: Pct Change due to behavior,-0.13,0.03,0.27,0.45,0.62,0.82,1.03,1.29,1.60,1.99,0.80,2.14
2,IIT: Pct Change due to macro,1.38,2.62,3.90,5.21,6.57,7.96,9.38,10.85,12.35,13.89,7.43,14.03
3,IIT: Overall Pct Change in taxes,-7.40,-6.12,-4.71,-3.34,-1.93,-0.46,1.08,2.69,4.40,6.23,-0.97,6.52
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,4.81,9.38,14.27,19.18,24.25,29.49,34.91,40.52,46.36,52.47,27.56,54.09
6,CIT: Pct Change due to macro,-3.42,-5.85,-8.29,-10.59,-12.79,-14.92,-16.95,-18.90,-20.77,-22.56,-14.23,-24.71
7,CIT: Overall Pct Change in taxes,1.22,2.98,4.79,6.56,8.36,10.18,12.04,13.96,15.96,18.07,9.41,16.02
8,All: Pct Change due to tax rates,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.06,-8.05
9,All: Pct Change due to behavior,0.17,0.60,1.12,1.58,2.06,2.56,3.09,3.68,4.33,5.06,2.42,5.35


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.41,-0.43,-0.46,-0.48,-0.49,-0.51,-0.54,-0.56,-0.58,-0.60,-5.06
9,Rev Change Due to Behavior,0.01,0.03,0.06,0.09,0.13,0.16,0.21,0.25,0.31,0.38,1.64
10,Rev Change Due to Macro,0.05,0.11,0.18,0.24,0.31,0.39,0.49,0.58,0.68,0.80,3.84
11,Total Revenue Change,-0.35,-0.30,-0.24,-0.17,-0.08,0.01,0.11,0.23,0.36,0.51,0.09


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.40,-0.43,-0.44,-0.46,-0.48,-0.49,-0.51,-0.53,-0.55,-4.66
1,Rev Change Due to Behavior,-0.01,0.00,0.01,0.02,0.03,0.05,0.06,0.08,0.10,0.13,0.48
2,Rev Change Due to Macro,0.06,0.12,0.20,0.27,0.35,0.44,0.54,0.65,0.77,0.90,4.30
3,Total Revenue Change,-0.32,-0.28,-0.24,-0.17,-0.10,-0.03,0.06,0.16,0.27,0.40,-0.24


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.37,-0.42,-0.43,-0.45,-0.47,-0.49,-0.51,-0.53,-0.55,-0.58,-4.81
1,Rev Change Due to Behavior,-0.01,0.00,0.01,0.02,0.03,0.05,0.06,0.08,0.10,0.13,0.49
2,Rev Change Due to Macro,0.06,0.13,0.20,0.28,0.36,0.46,0.56,0.68,0.80,0.94,4.46
3,Total Revenue Change,-0.32,-0.30,-0.24,-0.18,-0.11,-0.03,0.06,0.17,0.29,0.42,-0.23


In [8]:
# jason's get-around (with weifeng's correction in line 6)

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * (tc_reform + df_levels.loc[1, df_levels.columns[1:]])
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.32,-0.33,-0.34,-0.35,-0.36,-0.37,-0.38,-0.39,-0.41,-3.27
1,Rev Change Due to Behavior,-0.01,0.00,0.01,0.02,0.03,0.04,0.06,0.08,0.10,0.13,0.46
2,Rev Change Due to Macro,0.06,0.12,0.19,0.26,0.34,0.43,0.53,0.64,0.76,0.90,4.23
3,Total Revenue Change,0.05,-0.20,-0.13,-0.06,0.02,0.11,0.22,0.33,0.47,0.62,1.42


In [9]:
df_levels.to_csv('og_usa_result_w_tcja_prod_1.01.csv')